# Extreme Points of the Fixed-Point Polytope

This notebook computes the number of extreme points of the $n$-gram fixed-point polytope $\mathcal{F}_n$ for various alphabet sizes $s$ and model orders $n$.

By Theorem 1(b), the extreme points are in bijection with simple directed cycles in the de Bruijn graph $B(n{-}1, s)$. Each extreme point is the uniform distribution on the $n$-grams traversed by a single deterministic periodic orbit.

**Reference:** Maurer, U. M. (1992). Asymptotically-tight bounds on the number of cycles in generalized de Bruijn–Good graphs. *Discrete Applied Mathematics*, 37/38, 421–436.

In [ ]:
import itertools
from collections import defaultdict
import time
import pandas as pd

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print("networkx not found; install with: pip install networkx")

## 1. Build the de Bruijn graph $B(m, s)$

In [ ]:
def build_debruijn(m, s):
    """
    Build the de Bruijn graph B(m, s) as a NetworkX DiGraph.
    
    Nodes: all m-grams over alphabet {0, ..., s-1}
    Edges: for each node (w_1, ..., w_m) and each symbol a in {0,...,s-1},
           there is an edge to (w_2, ..., w_m, a).
    
    Each edge corresponds to an (m+1)-gram = n-gram.
    """
    G = nx.DiGraph()
    alphabet = list(range(s))
    if m == 0:
        # B(0, s): single node with s self-loops
        node = ()
        for a in alphabet:
            G.add_edge(node, node, label=a)
        return G
    for node in itertools.product(alphabet, repeat=m):
        for a in alphabet:
            successor = node[1:] + (a,)
            G.add_edge(node, successor)
    return G

## 2. Count simple directed cycles

We use NetworkX's `simple_cycles` (Johnson's algorithm) for exact enumeration on small graphs.

In [ ]:
def count_simple_cycles(m, s, timeout=60):
    """
    Count all simple directed cycles in B(m, s).
    Returns (count, elapsed_seconds) or (None, elapsed) if timeout exceeded.
    """
    G = build_debruijn(m, s)
    t0 = time.time()
    count = 0
    for _ in nx.simple_cycles(G):
        count += 1
        if time.time() - t0 > timeout:
            return None, time.time() - t0  # timed out
    return count, time.time() - t0


def count_cycles_by_length(m, s, timeout=60):
    """
    Count simple directed cycles in B(m, s) grouped by length.
    Returns dict {length: count} and elapsed time.
    """
    G = build_debruijn(m, s)
    t0 = time.time()
    counts = defaultdict(int)
    for cycle in nx.simple_cycles(G):
        counts[len(cycle)] += 1
        if time.time() - t0 > timeout:
            return None, time.time() - t0
    return dict(sorted(counts.items())), time.time() - t0

## 3. Dimension formula

The dimension of the fixed-point polytope $\mathcal{F}_n$ is $s^{n-1}(s-1)$.

In [ ]:
def polytope_dim(n, s):
    """Dimension of the circulation polytope = s^{n-1} * (s-1)."""
    return (s ** (n - 1)) * (s - 1)

## 4. Compute the table

We compute the number of extreme points for all feasible combinations of $s$ (alphabet size) and $n$ (model order), with a timeout per case.

In [ ]:
# Parameters to explore
cases = []
for s in [2, 3, 4, 5, 6, 7, 8, 10]:
    for n in range(2, 9):
        m = n - 1
        num_nodes = s ** m
        # Only attempt enumeration for small enough graphs
        if num_nodes <= 32:
            cases.append((s, n))

print(f"Will attempt {len(cases)} cases.")
print(f"{'s':>3} {'n':>3} {'nodes':>7} {'edges':>9} {'dim':>7}  status")
print("-" * 55)

results = []
for s, n in cases:
    m = n - 1
    num_nodes = s ** m
    num_edges = s ** n
    dim = polytope_dim(n, s)
    
    nc, elapsed = count_simple_cycles(m, s, timeout=120)
    
    if nc is not None:
        print(f"{s:>3} {n:>3} {num_nodes:>7} {num_edges:>9,} {dim:>7}  {nc:>12,} cycles  ({elapsed:.1f}s)")
        results.append({
            's': s, 'n': n, 'nodes': num_nodes, 'edges': num_edges,
            'dim': dim, 'extreme_points': nc, 'time_s': round(elapsed, 2)
        })
    else:
        print(f"{s:>3} {n:>3} {num_nodes:>7} {num_edges:>9,} {dim:>7}  TIMEOUT ({elapsed:.0f}s)")
        results.append({
            's': s, 'n': n, 'nodes': num_nodes, 'edges': num_edges,
            'dim': dim, 'extreme_points': None, 'time_s': round(elapsed, 2)
        })

## 5. Display as a table

In [ ]:
df = pd.DataFrame(results)
df_display = df[['s', 'n', 'nodes', 'edges', 'dim', 'extreme_points', 'time_s']].copy()
df_display.columns = ['Alphabet s', 'Order n', '|Nodes| = s^(n-1)', '|Edges| = s^n',
                       'dim F_n', '# Extreme points', 'Time (s)']
df_display

## 6. Pivot table: $s$ vs $n$

In [ ]:
# Create a pivot table with s as rows and n as columns
pivot = df.pivot(index='s', columns='n', values='extreme_points')
pivot.index.name = 'Alphabet size s'
pivot.columns.name = 'Model order n'

# Format: integers where available, dash where not
def fmt(x):
    if pd.isna(x):
        return '—'
    return f'{int(x):,}'

print("Number of extreme points of the fixed-point polytope F_n")
print("(= simple directed cycles in the de Bruijn graph B(n-1, s))")
print()
print(pivot.to_string(formatters={col: fmt for col in pivot.columns}))

## 7. Cycle length distributions for small cases

For a few illustrative cases, we show how many cycles exist at each length.

In [ ]:
small_cases = [(2, 2), (2, 3), (2, 4), (2, 5), (3, 2), (3, 3), (4, 2)]

for s, n in small_cases:
    m = n - 1
    by_length, elapsed = count_cycles_by_length(m, s, timeout=120)
    if by_length is not None:
        total = sum(by_length.values())
        print(f"\nB({m}, {s})  [n={n}, s={s}]  — {total} cycles total:")
        for length, count in by_length.items():
            print(f"  length {length:>3}: {count:>8,} cycles")
    else:
        print(f"\nB({m}, {s}) timed out.")

## 8. Hamiltonian cycle counts (exact formula)

The number of Hamiltonian cycles in $B(k, s)$ (equivalently, de Bruijn sequences of order $k$) is given by the BEST theorem:

$$C_{k,s}(s^k) = \frac{(s!)^{s^{k-1}}}{s^k}$$

This provides a *lower bound* on the total number of extreme points, since Hamiltonian cycles are just one cycle length.

In [ ]:
import math

def hamiltonian_count(k, s):
    """Number of Hamiltonian cycles in B(k, s) via the BEST theorem."""
    return math.factorial(s) ** (s ** (k - 1)) // (s ** k)

print("Hamiltonian cycle counts (lower bound on total extreme points):")
print(f"{'s':>3} {'n':>3} {'k=n-1':>5}  {'Hamiltonian cycles':>30}  {'log10':>8}")
print("-" * 60)

for s in [2, 3, 4, 5, 10]:
    for n in [2, 3, 4, 5]:
        k = n - 1
        try:
            hc = hamiltonian_count(k, s)
            if hc < 10**15:
                print(f"{s:>3} {n:>3} {k:>5}  {hc:>30,}  {math.log10(hc):>8.1f}")
            else:
                print(f"{s:>3} {n:>3} {k:>5}  {'(astronomically large)':>30}  {math.log10(hc):>8.1f}")
        except (OverflowError, ValueError):
            print(f"{s:>3} {n:>3} {k:>5}  {'overflow':>30}")

## 9. LaTeX table for the appendix

Generate a LaTeX-formatted table suitable for inclusion in the appendix.

In [ ]:
print(r"\begin{tabular}{@{}ccccc@{}}")
print(r"\toprule")
print(r"$s$ & $n$ & $|\mathcal{C}| = s^{n-1}$ & $\dim \mathcal{F}_n$ & \# extreme points \\")
print(r"\midrule")

prev_s = None
for _, row in df.iterrows():
    if row['extreme_points'] is None:
        continue
    s_val = int(row['s'])
    if prev_s is not None and s_val != prev_s:
        print(r"\midrule")
    prev_s = s_val
    ep = int(row['extreme_points'])
    print(f"${s_val}$ & ${int(row['n'])}$ & ${int(row['nodes'])}$ & "
          f"${int(row['dim'])}$ & ${ep:,}$ \\")

print(r"\bottomrule")
print(r"\end{tabular}")

## 10. Summary

**Key observations:**

1. The number of extreme points grows super-exponentially in both $s$ and $n$.
2. For realistic vocabularies (e.g. $s = 50{,}000$ subword tokens) even at $n = 2$, the number of extreme points is astronomically large.
3. Each fixed point in the convex polytope is a mixture of these extremes, parameterised by continuously many mixture weights — so the space of stable distributions is vast.
4. Maurer (1992) provides asymptotically tight bounds showing that the number of length-$\ell$ cycles transitions sharply around $k \approx 2\log_s \ell$.